# Person 3 — Content-based and hybrid recommenders

**Role:** Use item metadata to build a **content-based** recommender, then combine it with collaborative signals in **hybrid** systems, including explicit **cold-start** handling.

## How it works (CEO-simple)

- **Content-based:** “If you liked games about X, here are other games whose descriptions and metadata look like X.” This works even when almost nobody has rated a product yet, as long as we have text/metadata.
- **Hybrid:** Blend **who bought what** (collaborative) with **what the product is about** (content), so we are less blind when interactions are thin.
- **Cold-start:** **New users** (no history) get **popular** unseen items; **sparse users** use **more content** via the **switching** hybrid; **new items** can still be recommended from metadata via the content channel.

## Inputs (from Person 1)

From `data/processed/`:

- `train.csv`, `val.csv`, `metadata_clean.csv` (`item_id`, `content_text` built from title, description, features, brand, category, etc.)

**Do not** use `test.csv` here — reserved for Person 4’s final comparison.

In [ ]:
from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Set, Tuple

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.preprocessing import MinMaxScaler

DEFAULT_RELEVANCE_THRESHOLD = 4.0
DEFAULT_K = 10


In [ ]:
# Resolve repo root: run from repo root or from notebooks/
_cwd = Path.cwd()
if (_cwd / "data" / "processed").exists():
    REPO_ROOT = _cwd
elif (_cwd.parent / "data" / "processed").exists():
    REPO_ROOT = _cwd.parent
else:
    REPO_ROOT = _cwd

PROC_DIR = REPO_ROOT / "data" / "processed"
OUTPUT_DIR = REPO_ROOT / "models" / "person3_outputs"

K = 10
ALPHA_CF = 0.6
MIN_HISTORY_FOR_CF = 5

# Content features: "tfidf" (default), "bow" (bag-of-words), or "tfidf" with lemmatization below
CONTENT_VECTORIZER = "tfidf"  # primary pipeline for hybrid models
MAX_FEATURES = 30000
NGRAM_RANGE = (1, 2)
USE_LEMMATIZATION = False  # set True for lemmatized TF-IDF (requires nltk data download)

# Optional extra content-only baselines (slower on large data; set True to compare)
RUN_CONTENT_BOW_BASELINE = False
RUN_CONTENT_LEMMA_BASELINE = False

print("REPO_ROOT:", REPO_ROOT.resolve())
print("PROC_DIR:", PROC_DIR.resolve())


## Review item metadata

Person 1 builds **`content_text`** by combining title, description, features, brand, and categories into one field. That is our **item profile** for TF-IDF / BoW.

We still inspect raw columns (when present) to document what signals feed the profile and to catch empty text early.

In [ ]:
_meta_path = PROC_DIR / "metadata_clean.csv"
if not _meta_path.exists():
    raise FileNotFoundError(f"Expected {_meta_path}")

meta_raw = pd.read_csv(_meta_path)
print("Shape:", meta_raw.shape)
print("Columns:", list(meta_raw.columns))
print(meta_raw.head(2).to_string())
if "content_text" in meta_raw.columns:
    lens = meta_raw["content_text"].fillna("").astype(str).str.len()
    print(
        "\ncontent_text length — min/median/max:",
        int(lens.min()),
        float(lens.median()),
        int(lens.max()),
    )
    empty = (lens == 0).mean()
    print(f"Share of rows with empty content_text: {empty:.2%}")
del meta_raw

## Implementation

**Feature engineering pipeline:** `_clean_text` (lowercase, strip punctuation, remove English stopwords) → optional **WordNet lemmatization** → **TF-IDF** (default) or **count BoW** via scikit-learn → **cosine similarity** between item vectors (`linear_kernel` on L2-normalized TF-IDF gives cosine).

**Optional advanced text models:** Dense embeddings (e.g. BERT, sentence-transformers) can replace the sparse matrix in `fit_content`, but add heavy dependencies and runtime; only add them if they clearly beat TF-IDF on validation.

Metrics, recommender class, and helpers:

In [ ]:
def _clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = [tok for tok in text.split() if tok and tok not in ENGLISH_STOP_WORDS]
    return " ".join(tokens)


def _lemmatize_corpus(series: pd.Series) -> pd.Series:
    """Optional: WordNet lemmas (uses nltk; downloads corpora on first use)."""
    try:
        import nltk
        from nltk.stem import WordNetLemmatizer

        nltk.download("wordnet", quiet=True)
        nltk.download("omw-1.4", quiet=True)
        lem = WordNetLemmatizer()

        def _lemmatize_one(s: str) -> str:
            if not s:
                return ""
            return " ".join(lem.lemmatize(tok) for tok in str(s).split())

        return series.map(_lemmatize_one)
    except Exception as exc:
        print("Lemmatization skipped:", exc)
        return series


def precision_at_k(recommended: List[str], relevant: Set[str], k: int) -> float:
    if k <= 0:
        return 0.0
    rec_set = set(recommended[:k])
    return len(rec_set.intersection(relevant)) / k


def recall_at_k(recommended: List[str], relevant: Set[str], k: int) -> float:
    if not relevant:
        return 0.0
    rec_set = set(recommended[:k])
    return len(rec_set.intersection(relevant)) / len(relevant)


def ndcg_at_k(recommended: List[str], relevant: Set[str], k: int) -> float:
    if not relevant:
        return 0.0
    recs = recommended[:k]
    discounts = np.log2(np.arange(2, len(recs) + 2))
    gains = np.array([1.0 if item in relevant else 0.0 for item in recs], dtype=float)
    dcg = np.sum(gains / discounts)

    ideal_len = min(len(relevant), k)
    ideal_discounts = np.log2(np.arange(2, ideal_len + 2))
    idcg = np.sum(np.ones(ideal_len) / ideal_discounts)
    return float(dcg / idcg) if idcg > 0 else 0.0


def ap_at_k(recommended: List[str], relevant: Set[str], k: int) -> float:
    if not relevant:
        return 0.0
    hits = 0
    sum_precision = 0.0
    for i, item in enumerate(recommended[:k]):
        if item in relevant:
            hits += 1
            sum_precision += hits / (i + 1)
    return sum_precision / min(len(relevant), k)


@dataclass
class DataBundle:
    train: pd.DataFrame
    val: pd.DataFrame
    metadata: pd.DataFrame


class ContentHybridRecommender:
    def __init__(
        self,
        data: DataBundle,
        relevance_threshold: float = DEFAULT_RELEVANCE_THRESHOLD,
    ) -> None:
        self.train = data.train.copy()
        self.val = data.val.copy()
        self.metadata = data.metadata.copy()
        self.relevance_threshold = relevance_threshold

        self._validate_required_columns()
        self._prepare_ids_as_str()

        self.item_ids: np.ndarray = self.metadata["item_id"].values
        self.item_id_to_idx: Dict[str, int] = {iid: i for i, iid in enumerate(self.item_ids)}
        self.user_seen_train: Dict[str, Set[str]] = (
            self.train.groupby("user_id")["item_id"].apply(set).to_dict()
        )
        self.user_train_count: Dict[str, int] = self.train.groupby("user_id").size().to_dict()
        self.user_val_relevant: Dict[str, Set[str]] = (
            self.val[self.val["rating"] >= self.relevance_threshold]
            .groupby("user_id")["item_id"]
            .apply(set)
            .to_dict()
        )
        self.eval_users: List[str] = sorted(
            [u for u, rel in self.user_val_relevant.items() if rel and u in self.user_seen_train]
        )

        self.popular_items: List[str] = (
            self.train.groupby("item_id")
            .size()
            .sort_values(ascending=False)
            .index.astype(str)
            .tolist()
        )

        self.tfidf_vectorizer: TfidfVectorizer | None = None
        self.item_tfidf: csr_matrix | None = None
        self.content_sim: np.ndarray | None = None
        self.item_cf_sim: np.ndarray | None = None

    def _validate_required_columns(self) -> None:
        train_req = {"user_id", "item_id", "rating"}
        val_req = {"user_id", "item_id", "rating"}
        meta_req = {"item_id"}
        if "content_text" not in self.metadata.columns:
            missing_meta = meta_req.union({"content_text"}) - set(self.metadata.columns)
            raise ValueError(
                f"metadata_clean.csv missing required columns: {sorted(missing_meta)}. "
                "Expected at least ['item_id', 'content_text']."
            )

        missing_train = train_req - set(self.train.columns)
        missing_val = val_req - set(self.val.columns)
        if missing_train:
            raise ValueError(f"train.csv missing required columns: {sorted(missing_train)}")
        if missing_val:
            raise ValueError(f"val.csv missing required columns: {sorted(missing_val)}")

    def _prepare_ids_as_str(self) -> None:
        for df in (self.train, self.val, self.metadata):
            df["user_id"] = df["user_id"].astype(str) if "user_id" in df.columns else None
            df["item_id"] = df["item_id"].astype(str)
        self.metadata["content_text"] = (
            self.metadata["content_text"].fillna("").astype(str).map(_clean_text)
        )

    def fit_content(
        self,
        max_features: int = 30000,
        ngram_range: Tuple[int, int] = (1, 2),
        vectorizer: str = "tfidf",
        use_lemmatization: bool = False,
    ) -> None:
        """Build item–item content similarity using TF-IDF or bag-of-words + cosine (linear_kernel on L2-normalized TF-IDF)."""
        corpus = self.metadata["content_text"]
        if use_lemmatization:
            corpus = _lemmatize_corpus(corpus)
        kind = (vectorizer or "tfidf").lower()
        if kind == "tfidf":
            self.tfidf_vectorizer = TfidfVectorizer(max_features=max_features, ngram_range=ngram_range)
        elif kind == "bow":
            self.tfidf_vectorizer = CountVectorizer(max_features=max_features, ngram_range=ngram_range)
        else:
            raise ValueError("vectorizer must be 'tfidf' or 'bow'")
        self.item_tfidf = self.tfidf_vectorizer.fit_transform(corpus)
        self.content_sim = linear_kernel(self.item_tfidf, self.item_tfidf)

    def fit_item_cf(self) -> None:
        users = self.train["user_id"].astype("category")
        items = self.train["item_id"].astype("category")
        rows = users.cat.codes.values
        cols = items.cat.codes.values
        vals = np.ones(len(self.train), dtype=np.float32)

        user_item = csr_matrix((vals, (rows, cols)), shape=(users.cat.categories.size, items.cat.categories.size))
        item_user = user_item.T
        cf_sim = linear_kernel(item_user, item_user)

        item_order = items.cat.categories.astype(str).tolist()
        cf_index = {iid: idx for idx, iid in enumerate(item_order)}

        full_sim = np.zeros((len(self.item_ids), len(self.item_ids)), dtype=np.float32)
        for i, iid_i in enumerate(self.item_ids):
            if iid_i not in cf_index:
                continue
            row_i = cf_index[iid_i]
            for j, iid_j in enumerate(self.item_ids):
                if iid_j not in cf_index:
                    continue
                full_sim[i, j] = cf_sim[row_i, cf_index[iid_j]]
        self.item_cf_sim = full_sim

    def _top_popular_unseen(self, seen: Set[str], k: int) -> List[str]:
        return [item for item in self.popular_items if item not in seen][:k]

    def recommend_content(self, user_id: str, k: int = DEFAULT_K) -> List[str]:
        if self.content_sim is None:
            raise RuntimeError("fit_content() must be called before recommend_content().")
        seen = self.user_seen_train.get(user_id, set())
        if not seen:
            return self._top_popular_unseen(seen, k)

        seen_idx = [self.item_id_to_idx[i] for i in seen if i in self.item_id_to_idx]
        if not seen_idx:
            return self._top_popular_unseen(seen, k)

        content_scores = self.content_sim[seen_idx].mean(axis=0)
        ranked_idx = np.argsort(content_scores)[::-1]
        recs: List[str] = []
        for idx in ranked_idx:
            iid = self.item_ids[idx]
            if iid in seen:
                continue
            recs.append(iid)
            if len(recs) >= k:
                break
        if len(recs) < k:
            recs.extend(self._top_popular_unseen(seen.union(set(recs)), k - len(recs)))
        return recs

    def recommend_weighted_hybrid(
        self,
        user_id: str,
        k: int = DEFAULT_K,
        alpha_cf: float = 0.6,
    ) -> List[str]:
        if self.content_sim is None or self.item_cf_sim is None:
            raise RuntimeError("fit_content() and fit_item_cf() must be called before hybrid recommendations.")
        seen = self.user_seen_train.get(user_id, set())
        if not seen:
            return self._top_popular_unseen(seen, k)

        seen_idx = [self.item_id_to_idx[i] for i in seen if i in self.item_id_to_idx]
        if not seen_idx:
            return self._top_popular_unseen(seen, k)

        content_scores = self.content_sim[seen_idx].mean(axis=0).reshape(-1, 1)
        cf_scores = self.item_cf_sim[seen_idx].mean(axis=0).reshape(-1, 1)

        scaler = MinMaxScaler()
        content_norm = scaler.fit_transform(content_scores).ravel()
        cf_norm = scaler.fit_transform(cf_scores).ravel()

        hybrid_scores = alpha_cf * cf_norm + (1.0 - alpha_cf) * content_norm
        ranked_idx = np.argsort(hybrid_scores)[::-1]

        recs: List[str] = []
        for idx in ranked_idx:
            iid = self.item_ids[idx]
            if iid in seen:
                continue
            recs.append(iid)
            if len(recs) >= k:
                break
        if len(recs) < k:
            recs.extend(self._top_popular_unseen(seen.union(set(recs)), k - len(recs)))
        return recs

    def recommend_switching_hybrid(
        self,
        user_id: str,
        k: int = DEFAULT_K,
        min_history_for_cf: int = 5,
        alpha_cf: float = 0.6,
    ) -> List[str]:
        history_len = self.user_train_count.get(user_id, 0)
        if history_len >= min_history_for_cf:
            return self.recommend_weighted_hybrid(user_id=user_id, k=k, alpha_cf=alpha_cf)
        return self.recommend_content(user_id=user_id, k=k)

    def evaluate(self, model_name: str, recommend_fn, k: int = DEFAULT_K) -> Dict[str, float]:
        precisions, recalls, ndcgs, maps = [], [], [], []
        all_recommended = set()

        for user_id in self.eval_users:
            relevant = self.user_val_relevant.get(user_id, set())
            recs = recommend_fn(user_id, k)
            all_recommended.update(recs)
            precisions.append(precision_at_k(recs, relevant, k))
            recalls.append(recall_at_k(recs, relevant, k))
            ndcgs.append(ndcg_at_k(recs, relevant, k))
            maps.append(ap_at_k(recs, relevant, k))

        coverage = len(all_recommended) / len(self.item_ids) if len(self.item_ids) else 0.0
        return {
            "model": model_name,
            "precision_at_k": float(np.mean(precisions)) if precisions else 0.0,
            "recall_at_k": float(np.mean(recalls)) if recalls else 0.0,
            "ndcg_at_k": float(np.mean(ndcgs)) if ndcgs else 0.0,
            "map_at_k": float(np.mean(maps)) if maps else 0.0,
            "coverage": coverage,
            "eval_users": len(self.eval_users),
        }
def _prepare_ids_safe(train: pd.DataFrame, val: pd.DataFrame, metadata: pd.DataFrame) -> None:
    for df in (train, val):
        df["user_id"] = df["user_id"].astype(str)
        df["item_id"] = df["item_id"].astype(str)
    metadata["item_id"] = metadata["item_id"].astype(str)
    metadata["content_text"] = (
        metadata["content_text"].fillna("").astype(str).map(_clean_text)
    )


class ContentHybridRecommenderNB(ContentHybridRecommender):
    def _prepare_ids_as_str(self) -> None:
        _prepare_ids_safe(self.train, self.val, self.metadata)


def load_data_nb(processed_dir: Path) -> DataBundle:
    train_path = processed_dir / "train.csv"
    val_path = processed_dir / "val.csv"
    metadata_path = processed_dir / "metadata_clean.csv"
    missing = [p.name for p in (train_path, val_path, metadata_path) if not p.exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing required files in {processed_dir}: {missing}. "
            "Run Person 1 preprocessing first, then rerun this notebook."
        )
    return DataBundle(
        train=pd.read_csv(train_path),
        val=pd.read_csv(val_path),
        metadata=pd.read_csv(metadata_path),
    )



## Fit, evaluate, save outputs

**Protocol (aligned with Person 2):** For each eligible user, recommend top-K **unseen** training items; **relevant** if the user rated that item ≥ 4 in **validation** (`val.csv`).

| Model | Logic |
| --- | --- |
| Content | Average item–item similarity (TF-IDF or BoW vectors + cosine) from items the user consumed in train. |
| Weighted hybrid | MinMax-normalize **content** scores and **item–item CF** co-interaction similarity from train; combine `ALPHA_CF * cf + (1 - ALPHA_CF) * content`. |
| Switching hybrid | If train interactions `< MIN_HISTORY_FOR_CF` → content path; else → weighted hybrid (helps **cold-start / sparse** users). |

**Cold-start summary:** No history → global **popularity** among unseen items; sparse history → switching steers toward content; **new items** appear in the catalog metadatum → still reachable via content similarity.

Outputs for Person 4: `person3_model_results.csv`, `person3_sample_recommendations.csv` under `models/person3_outputs/`.

In [ ]:
data = load_data_nb(PROC_DIR)

# Primary recommender: drives hybrid models (uses CONTENT_VECTORIZER + USE_LEMMATIZATION)
recommender = ContentHybridRecommenderNB(data=data)
recommender.fit_content(
    max_features=MAX_FEATURES,
    ngram_range=NGRAM_RANGE,
    vectorizer=CONTENT_VECTORIZER,
    use_lemmatization=USE_LEMMATIZATION,
)
recommender.fit_item_cf()

results = []

primary_content_name = f"content_{CONTENT_VECTORIZER}"
if USE_LEMMATIZATION:
    primary_content_name += "_lemmatized"

results.append(
    recommender.evaluate(
        model_name=primary_content_name,
        recommend_fn=lambda u, k, rr=recommender: rr.recommend_content(user_id=u, k=k),
        k=K,
    )
)

if RUN_CONTENT_BOW_BASELINE and CONTENT_VECTORIZER != "bow":
    r_bow = ContentHybridRecommenderNB(data=data)
    r_bow.fit_content(
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
        vectorizer="bow",
        use_lemmatization=False,
    )
    r_bow.fit_item_cf()
    results.append(
        r_bow.evaluate(
            model_name="content_bow_baseline",
            recommend_fn=lambda u, k, rr=r_bow: rr.recommend_content(user_id=u, k=k),
            k=K,
        )
    )

if RUN_CONTENT_LEMMA_BASELINE and not USE_LEMMATIZATION and CONTENT_VECTORIZER == "tfidf":
    r_lem = ContentHybridRecommenderNB(data=data)
    r_lem.fit_content(
        max_features=MAX_FEATURES,
        ngram_range=NGRAM_RANGE,
        vectorizer="tfidf",
        use_lemmatization=True,
    )
    r_lem.fit_item_cf()
    results.append(
        r_lem.evaluate(
            model_name="content_tfidf_lemmatized_baseline",
            recommend_fn=lambda u, k, rr=r_lem: rr.recommend_content(user_id=u, k=k),
            k=K,
        )
    )

hybrid_weighted_name = f"hybrid_weighted__{CONTENT_VECTORIZER}"
if USE_LEMMATIZATION:
    hybrid_weighted_name += "_lemmatized"

results.extend(
    [
        recommender.evaluate(
            model_name=hybrid_weighted_name,
            recommend_fn=lambda u, k, rr=recommender: rr.recommend_weighted_hybrid(
                user_id=u, k=k, alpha_cf=ALPHA_CF
            ),
            k=K,
        ),
        recommender.evaluate(
            model_name="hybrid_switching",
            recommend_fn=lambda u, k, rr=recommender: rr.recommend_switching_hybrid(
                user_id=u,
                k=k,
                min_history_for_cf=MIN_HISTORY_FOR_CF,
                alpha_cf=ALPHA_CF,
            ),
            k=K,
        ),
    ]
)

results_df = pd.DataFrame(results)

sample_users = recommender.eval_users[:10]
sample_rows = []
for u in sample_users:
    sample_rows.append(
        {
            "user_id": u,
            "content_recs": recommender.recommend_content(u, K),
            "weighted_hybrid_recs": recommender.recommend_weighted_hybrid(u, K, ALPHA_CF),
            "switching_hybrid_recs": recommender.recommend_switching_hybrid(
                u, K, MIN_HISTORY_FOR_CF, ALPHA_CF
            ),
        }
    )
sample_df = pd.DataFrame(sample_rows)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_csv(OUTPUT_DIR / "person3_model_results.csv", index=False)
sample_df.to_csv(OUTPUT_DIR / "person3_sample_recommendations.csv", index=False)

try:
    from IPython.display import display

    display(results_df.sort_values("ndcg_at_k", ascending=False))
except ImportError:
    print(results_df.sort_values("ndcg_at_k", ascending=False).to_string(index=False))

print("Saved:", OUTPUT_DIR / "person3_model_results.csv")
